# Direction Classification

## Objective 
Investigate whether technical features can predict the 5-day returns direction for AAPL 

Target Variable:
- 1 - Positive direction
- 0 - Zero or negative direction

This notebook develops and evaluates binary classification models

# TODO 

- XGBoost threshold tuning with selected features
- Hyperparameter tuning

In [436]:
import numpy as np
import pandas as pd
import yfinance as yf
from market_ml.data_loader import DataLoader
import matplotlib.pyplot as plt
import plotly.express as px
from market_ml.config import * 
from market_ml.EDA import * 
from market_ml.feature_engineering import *
from market_ml.models import *

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import *
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.inspection import permutation_importance


# Data Loading 


In [397]:
stock_dl = DataLoader(tickers=TICKERS, period="10y", interval="1d", file_name='stock_data.parquet')
benchmark_dl = DataLoader(tickers=[BENCHMARK_TICKERS], period="10y", interval="1d", file_name='benchmark_data.parquet')

market_data = stock_dl.load_data()
benchmark_data = benchmark_dl.load_data()
sectors = market_data["Sector"].unique()

benchmark_data

Data loaded for ['AAPL']
Data loaded for [['SPY', 'QQQ']]


Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-11-14,QQQ,NaN,107.121819,108.346019,106.561120,108.243224,35906000
1,2016-11-14,SPY,NaN,184.966873,185.547597,184.223901,185.342633,94580000
2,2016-11-15,QQQ,NaN,108.607681,108.981483,107.775976,107.934840,31796600
3,2016-11-15,SPY,NaN,186.410126,186.410126,185.146216,185.351167,91652600
4,2016-11-16,QQQ,NaN,109.252464,109.327219,108.093678,108.093678,16538100
...,...,...,...,...,...,...,...,...
5023,2026-11-11,SPY,NaN,759.809998,762.510010,754.900024,756.799988,39151000
5024,2026-11-12,QQQ,NaN,688.169983,694.559998,679.830017,680.320007,37219500
5025,2026-11-12,SPY,NaN,757.119995,763.039978,753.669983,753.979980,37460600
5026,2026-11-13,QQQ,NaN,679.890015,688.750000,671.679993,688.539978,41798500


In [398]:
engineer = FeatureEngineer(market_data, benchmark_data, filename="stock_data_with_features.parquet")
data  = engineer.load_features()

data.shape

Data loaded for Price       Date Ticker      Sector       Close        High         Low  \
0     2016-11-14   AAPL  Technology   24.280031   24.762370   23.905645   
1     2016-11-15   AAPL  Technology   24.601593   24.732514   24.383393   
2     2016-11-16   AAPL  Technology   25.263075   25.318201   24.484442   
3     2016-11-17   AAPL  Technology   25.253891   25.345765   24.996645   
4     2016-11-18   AAPL  Technology   25.279156   25.389405   25.187283   
...          ...    ...         ...         ...         ...         ...   
2509  2026-11-09   AAPL  Technology  350.423737  352.062476  344.518220   
2510  2026-11-10   AAPL  Technology  348.245392  353.601307  346.196930   
2511  2026-11-11   AAPL  Technology  355.359985  355.639772  349.064750   
2512  2026-11-12   AAPL  Technology  354.880005  359.369995  354.410004   
2513  2026-11-13   AAPL  Technology  356.269989  360.829987  355.750000   

Price        Open     Volume  
0       24.739402  204702000  
1       24.477563  12

(2449, 26)

In [399]:
print(data.shape)

print(data[["Date", "Ticker"]].head())

print(data.sort_values(["Date", "Ticker"]).reset_index(drop=True))



(2449, 26)
Price       Date Ticker
60    2017-02-10   AAPL
61    2017-02-13   AAPL
62    2017-02-14   AAPL
63    2017-02-15   AAPL
64    2017-02-16   AAPL
Price       Date Ticker      Sector       Close        High         Low  \
0     2017-02-10   AAPL  Technology   30.477581   30.666741   30.461435   
1     2017-02-13   AAPL  Technology   30.747471   30.869735   30.622905   
2     2017-02-14   AAPL  Technology   31.146549   31.162695   30.738243   
3     2017-02-15   AAPL  Technology   31.259592   31.434912   31.054286   
4     2017-02-16   AAPL  Technology   31.222681   31.349553   31.105031   
...          ...    ...         ...         ...         ...         ...   
2444  2026-11-02   AAPL  Technology  338.063141  343.538961  335.565039   
2445  2026-11-03   AAPL  Technology  331.648010  334.246042  325.082991   
2446  2026-11-04   AAPL  Technology  349.244659  354.230861  340.731108   
2447  2026-11-05   AAPL  Technology  343.908691  355.110171  341.800278   
2448  2026-11-06   A

In [400]:
print("Start:", data["Date"].min())
print("End:  ", data["Date"].max())

Start: 2017-02-10 00:00:00
End:   2026-11-06 00:00:00


# Splitting Data into Train and Test 

In [401]:
train_data, validation_data, test_data = split_chronological_data(data)

In [402]:
print("Training:", train_data["Date"].min(), "to", train_data["Date"].max())
print("Validation:", validation_data["Date"].min(), "to", validation_data
      ["Date"].max())
print("Testing:", test_data["Date"].min(), "to", test_data["Date"].max())


Training: 2017-02-10 00:00:00 to 2023-11-24 00:00:00
Validation: 2023-12-04 00:00:00 to 2025-05-14 00:00:00
Testing: 2025-05-22 00:00:00 to 2026-11-06 00:00:00


In [403]:
for name, dataset in [
    ("Training", train_data),
    ("Validation", validation_data),
    ("Testing", test_data)
]:
    print(
        f"{name}: {dataset['Date'].min()} to "
        f"{dataset['Date'].max()} | "
        f"{len(dataset)} observations"
    )

Training: 2017-02-10 00:00:00 to 2023-11-24 00:00:00 | 1709 observations
Validation: 2023-12-04 00:00:00 to 2025-05-14 00:00:00 | 362 observations
Testing: 2025-05-22 00:00:00 to 2026-11-06 00:00:00 | 368 observations


In [404]:
#creating target and features column, also removing the future returns column so no future info is contained. 
target_column = "Target_5D_Direction"

excluded_columns = [
    "Target_5D_Direction",
    "Future_Return_5D",
    "Date",
    "Ticker",
    "Sector",
]

feature_columns = [
    col for col in data.columns
    if col not in excluded_columns
]


Splitting into X and y 

In [405]:
X_train = train_data[feature_columns]
y_train = train_data[target_column]

X_validation = validation_data[feature_columns]
y_validation = validation_data[target_column]

X_test = test_data[feature_columns]
y_test = test_data[target_column]

In [406]:
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_validation.shape, y_validation.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (1709, 21) (1709,)
Validation: (362, 21) (362,)
Testing: (368, 21) (368,)


# Plotting the target distribution for each dataset (Training, Validation and Testing)

In [407]:
class_data = pd.concat([
    y_train.rename("Direction").to_frame().assign(Dataset="Training"),
    y_validation.rename("Direction").to_frame().assign(Dataset="Validation"),
    y_test.rename("Direction").to_frame().assign(Dataset="Testing")
])

class_data["Direction"] = class_data["Direction"].map({
    0: "Down (Negative or no change)",
    1: "Up (Positive)"
})

fig = px.histogram(
    class_data,
    x="Dataset",
    color="Direction",
    barmode="group",
    histfunc="count",
    title="AAPL Five-Day Direction by Dataset"
)

fig.show()

Some Imbalance, seeing more positive than negative (may address this at a later stage)

In [408]:
print("Original data:", len(data))
print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Testing:", len(test_data))

print("Missing targets:", data[target_column].isna().sum())

Original data: 2449
Training: 1709
Validation: 362
Testing: 368
Missing targets: 0


# A Simple Majority Class Baseline Model

In [409]:
majority_class = y_train.mode()[0]

print(majority_class)

1


In [410]:
baseline_predictions = np.full(len(y_validation),majority_class)
baseline_probabilities = np.ones(len(y_validation))

In [411]:
model_comparison_df = pd.DataFrame(columns=["Model", "Accuracy", "Balanced_accuracy", "Precision", "Recall", "ROC_AUC", "PR_AUC"])
print("Majority class:", majority_class)

accuracy = np.round(accuracy_score(y_validation, baseline_predictions), 3)
print(f"Accuracy: {accuracy}")

balanced_accuracy = np.round(balanced_accuracy_score(y_validation, baseline_predictions), 3)
print(f"Balanced accuracy: {balanced_accuracy}")

precision = np.round(precision_score(y_validation, baseline_predictions), 3)
recall = np.round(recall_score(y_validation, baseline_predictions), 3)

roc_auc = np.round(roc_auc_score(y_validation, baseline_probabilities),3)

pr_auc = np.round(average_precision_score(y_validation, baseline_probabilities),3)

print("\nClassification report:")
print(classification_report(
    y_validation,
    baseline_predictions,
    zero_division=0
))


baseline_metrics = ['Baseline', accuracy, balanced_accuracy, precision, recall, roc_auc, pr_auc ]

if "Baseline" not in model_comparison_df["Model"].values:
    model_comparison_df.loc[len(model_comparison_df)] = baseline_metrics
model_comparison_df

Majority class: 1
Accuracy: 0.53
Balanced accuracy: 0.5

Classification report:
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       170
         1.0       0.53      1.00      0.69       192

    accuracy                           0.53       362
   macro avg       0.27      0.50      0.35       362
weighted avg       0.28      0.53      0.37       362



,Model,Accuracy,Balanced_accuracy,Precision,Recall,ROC_AUC,PR_AUC
0,Baseline,0.53,0.5,0.53,1.0,0.5,0.53


As expected (no suprises here!)

# Logistic Regression

As the the features have wildly different magnitudes, we need to use a standard scaler here for logistic regression. Logistic regression uses weights and bigger numbers have a bigger influence over the weights, therefore we need to scale. 

In [412]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train) #learns the scaling parameters

X_validation_scaled = scaler.transform(X_validation) #Applies the scaling parameters

In [413]:

logistic_model = LogisticRegression() #defining the model

logistic_model.fit(X_train_scaled, y_train) #training the model


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [414]:
y_validation_pred_lr = logistic_model.predict(X_validation_scaled) #generating predictions

In [415]:
y_validation_prob_lr = logistic_model.predict_proba(X_validation_scaled) #generating probabilities. 
positive_probabilities_lr = y_validation_prob_lr[:, 1]

In [416]:
#Metrics 

accuracy = np.round(accuracy_score(y_validation, y_validation_pred_lr), 3)
balanced_accuracy = np.round(balanced_accuracy_score(y_validation, y_validation_pred_lr), 3)
precision = np.round(precision_score(y_validation, y_validation_pred_lr), 3)
recall = np.round(recall_score(y_validation, y_validation_pred_lr), 3)

roc_auc = np.round(roc_auc_score(y_validation, positive_probabilities_lr),3)
pr_auc = np.round(average_precision_score(y_validation, positive_probabilities_lr),3)


print(f"Accuracy: {accuracy}")
print(f"Balanced accuracy: {balanced_accuracy}")
#print(f"Precision: {precision}")
#print(f"Recall: {recall}")
print(f"ROC-AUC: {roc_auc}")
print(f"PR_AUC: {pr_auc}")

print("\nClassification report:")
print(classification_report(
    y_validation,
    baseline_predictions,
    zero_division=0
))


log_metrics = ['Logistic Regression', accuracy, balanced_accuracy, precision, recall, roc_auc, pr_auc ]

if "Logistic Regression" not in model_comparison_df["Model"].values:
    model_comparison_df.loc[len(model_comparison_df)] = log_metrics
model_comparison_df

Accuracy: 0.494
Balanced accuracy: 0.492
ROC-AUC: 0.518
PR_AUC: 0.554

Classification report:
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       170
         1.0       0.53      1.00      0.69       192

    accuracy                           0.53       362
   macro avg       0.27      0.50      0.35       362
weighted avg       0.28      0.53      0.37       362



,Model,Accuracy,Balanced_accuracy,Precision,Recall,ROC_AUC,PR_AUC
0,Baseline,0.530,0.500,0.530,1.000,0.500,0.530
1,Logistic Regression,0.494,0.492,0.523,0.526,0.518,0.554


# Random Forest

In [417]:
random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total n

In [418]:
y_validation_pred_rf = random_forest_model.predict(X_validation)

In [419]:
y_validation_prob_rf = random_forest_model.predict_proba(X_validation) #generating probabilities. 
positive_probabilities_rf = y_validation_prob_rf[:, 1]

In [420]:
#Metrics 
accuracy = np.round(accuracy_score(y_validation, y_validation_pred_rf), 3)
balanced_accuracy = np.round(balanced_accuracy_score(y_validation, y_validation_pred_rf), 3)
precision = np.round(precision_score(y_validation, y_validation_pred_rf), 3)
recall = np.round(recall_score(y_validation, y_validation_pred_rf), 3)

roc_auc = np.round(roc_auc_score(y_validation, positive_probabilities_rf),3)
pr_auc = np.round(average_precision_score(y_validation, positive_probabilities_rf),3)


print(f"Accuracy: {accuracy}")
print(f"Balanced accuracy: {balanced_accuracy}")
#print(f"Precision: {precision}")
#print(f"Recall: {recall}")
print(f"ROC-AUC: {roc_auc}")
print(f"PR_AUC: {pr_auc}")

print("\nClassification report:")
print(classification_report(
    y_validation,
    baseline_predictions,
    zero_division=0
))


rf_metrics = ['Random Forest', accuracy, balanced_accuracy, precision, recall, roc_auc, pr_auc ]

if "Random Forest" not in model_comparison_df["Model"].values:
    model_comparison_df.loc[len(model_comparison_df)] = rf_metrics
model_comparison_df

Accuracy: 0.506
Balanced accuracy: 0.517
ROC-AUC: 0.516
PR_AUC: 0.58

Classification report:
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       170
         1.0       0.53      1.00      0.69       192

    accuracy                           0.53       362
   macro avg       0.27      0.50      0.35       362
weighted avg       0.28      0.53      0.37       362



,Model,Accuracy,Balanced_accuracy,Precision,Recall,ROC_AUC,PR_AUC
0,Baseline,0.530,0.500,0.530,1.000,0.500,0.530
1,Logistic Regression,0.494,0.492,0.523,0.526,0.518,0.554
2,Random Forest,0.506,0.517,0.557,0.333,0.516,0.580


initial random forest model doing better than the logistic regression but not as good as the baseline

# XGBoost

In [421]:
xgboost_model = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgboost_model.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [422]:
y_validation_pred_xgb = xgboost_model.predict(X_validation)

In [423]:
y_validation_prob_xgb = xgboost_model.predict_proba(X_validation) #generating probabilities. 
positive_probabilities_xgb = y_validation_prob_xgb[:, 1]

In [424]:
#Metrics 
accuracy = np.round(accuracy_score(y_validation, y_validation_pred_xgb), 3)
balanced_accuracy = np.round(balanced_accuracy_score(y_validation, y_validation_pred_xgb), 3)
precision = np.round(precision_score(y_validation, y_validation_pred_xgb), 3)
recall = np.round(recall_score(y_validation, y_validation_pred_xgb), 3)

roc_auc = np.round(roc_auc_score(y_validation, positive_probabilities_xgb),3)
pr_auc = np.round(average_precision_score(y_validation, positive_probabilities_xgb),3)


print(f"Accuracy: {accuracy}")
print(f"Balanced accuracy: {balanced_accuracy}")
#print(f"Precision: {precision}")
#print(f"Recall: {recall}")
print(f"ROC-AUC: {roc_auc}")
print(f"PR_AUC: {pr_auc}")

print("\nClassification report:")
print(classification_report(
    y_validation,
    y_validation_pred_xgb,
    zero_division=0
))


xgb_metrics = ['XGBoost', accuracy, balanced_accuracy, precision, recall, roc_auc, pr_auc ]

if "XGBoost" not in model_comparison_df["Model"].values:
    model_comparison_df.loc[len(model_comparison_df)] = xgb_metrics
model_comparison_df

Accuracy: 0.544
Balanced accuracy: 0.563
ROC-AUC: 0.555
PR_AUC: 0.625

Classification report:
              precision    recall  f1-score   support

         0.0       0.51      0.88      0.64       170
         1.0       0.70      0.25      0.37       192

    accuracy                           0.54       362
   macro avg       0.60      0.56      0.51       362
weighted avg       0.61      0.54      0.50       362



,Model,Accuracy,Balanced_accuracy,Precision,Recall,ROC_AUC,PR_AUC
0,Baseline,0.530,0.500,0.530,1.000,0.500,0.530
1,Logistic Regression,0.494,0.492,0.523,0.526,0.518,0.554
2,Random Forest,0.506,0.517,0.557,0.333,0.516,0.580
3,XGBoost,0.544,0.563,0.696,0.250,0.555,0.625


Investigating XGBoost's low recall

In [425]:
print("Random Forest:")
print(confusion_matrix(y_validation, y_validation_pred_rf))

print("XGBoost:")
print(confusion_matrix(y_validation, y_validation_pred_xgb))

Random Forest:
[[119  51]
 [128  64]]
XGBoost:
[[149  21]
 [144  48]]


In [426]:

pd.Series(y_validation_prob_xgb[:, 1]).describe()

count    362.000000
mean       0.349012
std        0.186300
min        0.037737
25%        0.222464
50%        0.306072
75%        0.442045
max        0.899300
dtype: float64

Trying different threshold amounts: 

In [427]:
threshold_results = []

for threshold in np.arange(0.35, 0.56, 0.05):
    predictions = (y_validation_prob_xgb[:, 1] >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_validation, predictions, zero_division=0),
        "Recall": recall_score(y_validation, predictions, zero_division=0),
        "F1": f1_score(y_validation, predictions, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_validation, predictions)
    })

threshold_results = pd.DataFrame(threshold_results)
threshold_results.round(3)

,Threshold,Precision,Recall,F1,Balanced Accuracy
0,0.35,0.589,0.464,0.519,0.549
1,0.40,0.617,0.370,0.463,0.555
2,0.45,0.596,0.276,0.377,0.532
3,0.50,0.696,0.250,0.368,0.563
4,0.55,0.724,0.219,0.336,0.562


Threshold of 0.45 might be the best option as giving a significant increase in recall. 

In [428]:
y_validation_pred_xgb_045 = (
    y_validation_prob_xgb[:, 1] >= 0.45
).astype(int)

confusion_matrix(y_validation, y_validation_pred_xgb_045)

array([[134,  36],
       [139,  53]])

In [429]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgboost_model.feature_importances_
}).sort_values("Importance", ascending=False)

feature_importance

,Feature,Importance
1,High,0.060625
15,SMA_20D,0.058787
18,Price_to_SMA_50D,0.055122
17,Price_to_SMA_20D,0.054197
0,Close,0.052657
16,SMA_50D,0.052628
19,Volume_Average_20D,0.051913
9,Return_10D,0.051652
12,Volatility_20D,0.051433
2,Low,0.051300


In [430]:
fig = px.bar(
    feature_importance.sort_values("Importance"),
    x="Importance",
    y="Feature",
    orientation="h",
    title="XGBoost Feature Importance",
    labels={"Importance": "Importance", "Feature": "Feature"}
)

fig.show()

# Let's think about simplifying the feature set

In [431]:
reduced_features = [
    col for col in feature_columns
    if col not in ["SMA_20D", "SMA_50D"] #these are too similar
]

X_train_reduced = X_train[reduced_features]
X_validation_reduced = X_validation[reduced_features]

As XGBoost was a our best model from our benchmark, let's build on this. 

In [432]:
xgboost_model_red = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgboost_model_red.fit(X_train_reduced, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [433]:
y_validation_pred_xgb_reduced = xgboost_model_red.predict(X_validation_reduced)

In [434]:
y_validation_prob_xgb_red =xgboost_model_red.predict_proba(X_validation_reduced) #generating probabilities. 
positive_probabilities_xgb_red = y_validation_prob_xgb_red[:, 1]

In [435]:
#Metrics 
accuracy = np.round(accuracy_score(y_validation, y_validation_pred_xgb_reduced), 3)
balanced_accuracy = np.round(balanced_accuracy_score(y_validation, y_validation_pred_xgb_reduced), 3)
precision = np.round(precision_score(y_validation, y_validation_pred_xgb_reduced), 3)
recall = np.round(recall_score(y_validation, y_validation_pred_xgb_reduced), 3)

roc_auc = np.round(roc_auc_score(y_validation, positive_probabilities_xgb_red),3)
pr_auc = np.round(average_precision_score(y_validation, positive_probabilities_xgb_red),3)


print(f"Accuracy: {accuracy}")
print(f"Balanced accuracy: {balanced_accuracy}")
#print(f"Precision: {precision}")
#print(f"Recall: {recall}")
print(f"ROC-AUC: {roc_auc}")
print(f"PR_AUC: {pr_auc}")

print("\nClassification report:")
print(classification_report(
    y_validation,
    y_validation_pred_xgb,
    zero_division=0
))


xgb_red_metrics = ['XGBoost Reduced', accuracy, balanced_accuracy, precision, recall, roc_auc, pr_auc ]

if "XGBoost Reduced" not in model_comparison_df["Model"].values:
    model_comparison_df.loc[len(model_comparison_df)] = xgb_red_metrics
model_comparison_df

Accuracy: 0.506
Balanced accuracy: 0.52
ROC-AUC: 0.496
PR_AUC: 0.576

Classification report:
              precision    recall  f1-score   support

         0.0       0.51      0.88      0.64       170
         1.0       0.70      0.25      0.37       192

    accuracy                           0.54       362
   macro avg       0.60      0.56      0.51       362
weighted avg       0.61      0.54      0.50       362



,Model,Accuracy,Balanced_accuracy,Precision,Recall,ROC_AUC,PR_AUC
0,Baseline,0.530,0.500,0.530,1.000,0.500,0.530
1,Logistic Regression,0.494,0.492,0.523,0.526,0.518,0.554
2,Random Forest,0.506,0.517,0.557,0.333,0.516,0.580
3,XGBoost,0.544,0.563,0.696,0.250,0.555,0.625
4,XGBoost Reduced,0.506,0.520,0.570,0.276,0.496,0.576


Reduced features XGBoost doing worse on all features. Will keep full feature set for now. 


# Feature Permutation

In [437]:
permutation_result = permutation_importance(
    xgboost_model,
    X_validation,
    y_validation,
    scoring="roc_auc",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

In [440]:
permutation_importance_df = pd.DataFrame({
    "Feature": X_validation.columns,
    "Importance": permutation_result.importances_mean,
    "Std": permutation_result.importances_std
}).sort_values("Importance", ascending=False)

permutation_importance_df

,Feature,Importance,Std
10,Return_20D,0.033885,0.005902
16,SMA_50D,0.033260,0.010109
11,Return_60D,0.012397,0.006369
18,Price_to_SMA_50D,0.011325,0.009640
17,Price_to_SMA_20D,0.009678,0.006036
19,Volume_Average_20D,0.005109,0.003559
4,Volume,0.004854,0.002735
6,SPY_Return,0.000743,0.002245
3,Open,0.000003,0.000153
1,High,-0.000112,0.001207


Only a few features are contributing. 

In [442]:
selected_features = [
    "Return_20D",
    "Return_60D",
    "SMA_50D",
    "Price_to_SMA_50D",
    "Price_to_SMA_20D",
    "Volume_Average_20D",
    "Volume",
    "SPY_Return"
]

In [443]:
X_train_selected = X_train[selected_features]
X_validation_selected = X_validation[selected_features]

In [448]:
xgboost_model_selected = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgboost_model_selected.fit(X_train_selected, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [449]:
y_validation_pred_xgb_selected = xgboost_model_selected.predict(X_validation_selected)

In [451]:
y_validation_prob_xgb_selected =xgboost_model_selected.predict_proba(X_validation_selected) #generating probabilities. 
positive_probabilities_xgb_selected = y_validation_prob_xgb_selected[:, 1]

In [452]:
#Metrics 
accuracy = np.round(accuracy_score(y_validation, y_validation_pred_xgb_selected), 3)
balanced_accuracy = np.round(balanced_accuracy_score(y_validation, y_validation_pred_xgb_selected), 3)
precision = np.round(precision_score(y_validation, y_validation_pred_xgb_selected), 3)
recall = np.round(recall_score(y_validation, y_validation_pred_xgb_selected), 3)

roc_auc = np.round(roc_auc_score(y_validation, positive_probabilities_xgb_selected),3)
pr_auc = np.round(average_precision_score(y_validation, positive_probabilities_xgb_selected),3)


print(f"Accuracy: {accuracy}")
print(f"Balanced accuracy: {balanced_accuracy}")
#print(f"Precision: {precision}")
#print(f"Recall: {recall}")
print(f"ROC-AUC: {roc_auc}")
print(f"PR_AUC: {pr_auc}")

print("\nClassification report:")
print(classification_report(
    y_validation,
    y_validation_pred_xgb,
    zero_division=0
))


xgb_selected_metrics = ['XGBoost Reduced Selected', accuracy, balanced_accuracy, precision, recall, roc_auc, pr_auc ]

if "XGBoost Reduced Selected" not in model_comparison_df["Model"].values:
    model_comparison_df.loc[len(model_comparison_df)] = xgb_selected_metrics
model_comparison_df

Accuracy: 0.583
Balanced accuracy: 0.6
ROC-AUC: 0.633
PR_AUC: 0.693

Classification report:
              precision    recall  f1-score   support

         0.0       0.51      0.88      0.64       170
         1.0       0.70      0.25      0.37       192

    accuracy                           0.54       362
   macro avg       0.60      0.56      0.51       362
weighted avg       0.61      0.54      0.50       362



,Model,Accuracy,Balanced_accuracy,Precision,Recall,ROC_AUC,PR_AUC
0,Baseline,0.530,0.500,0.530,1.000,0.500,0.530
1,Logistic Regression,0.494,0.492,0.523,0.526,0.518,0.554
2,Random Forest,0.506,0.517,0.557,0.333,0.516,0.580
3,XGBoost,0.544,0.563,0.696,0.250,0.555,0.625
4,XGBoost Reduced,0.506,0.520,0.570,0.276,0.496,0.576
5,XGBoost Reduced Selected,0.583,0.600,0.747,0.323,0.633,0.693


Showing some improvement but recall still very conservative

In [453]:
threshold_results = []

for threshold in np.arange(0.30, 0.71, 0.05):

    y_pred = (
        positive_probabilities_xgb_selected >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Precision": precision_score(y_validation, y_pred, zero_division=0),
        "Recall": recall_score(y_validation, y_pred, zero_division=0),
        "F1": f1_score(y_validation, y_pred, zero_division=0),
        "Balanced Accuracy": balanced_accuracy_score(y_validation, y_pred)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.round(3)

,Threshold,Precision,Recall,F1,Balanced Accuracy
0,0.30,0.621,0.521,0.567,0.581
1,0.35,0.642,0.448,0.528,0.583
2,0.40,0.667,0.385,0.488,0.584
3,0.45,0.684,0.339,0.453,0.581
4,0.50,0.747,0.323,0.451,0.600
5,0.55,0.787,0.307,0.442,0.607
6,0.60,0.797,0.266,0.398,0.595
7,0.65,0.812,0.203,0.325,0.575
8,0.70,0.893,0.130,0.227,0.556
